# HP_06_01_NB_BronzeToSilver — Loading

Bronze → Silver transformation, minimum blocking validations, upsert, and per-entity quarantine.
The separate DQ notebook writes the centralized audit and score. Run loading first, then DQ with identical parameters.


## 1. Imports

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from delta.tables import DeltaTable
from datetime import datetime, timezone
import uuid
import re


## 2. Pipeline Parameters

In [ ]:
# Default values are for interactive testing.
# Fabric Pipeline Notebook activity overrides these at runtime.
PipelineRunId = "test11"
EntityName = "bronze_patients"
SourceName = "SYNTHEA"

# PipelineRunId = ""
# EntityName = ""
# SourceName = ""


In [ ]:
SourceName=f"{SourceName} -Bronze" 

## 3. Source / Target Configuration

In [ ]:
BRONZE_LAKEHOUSE = "HP_01_LH_Health_Bronze"
BRONZE_SCHEMA = "bronze"
SILVER_LAKEHOUSE = "HP_02_LH_Health_Silver"

# EntityName is passed by the pipeline as the physical Bronze table name,
# for example: bronze_patients, bronze_encounters, bronze_conditions.
bronze_table = EntityName.lower().strip()

if not bronze_table.startswith("bronze_"):
    raise ValueError(
        f"EntityName must be a Bronze table name beginning with 'bronze_'. "
        f"Received: {EntityName}"
    )

# Logical entity name used by TRANSFORMERS and ENTITY_CONFIG.
entity = bronze_table.removeprefix("bronze_")
entity= entity.removesuffix("s")

bronze_full_table = f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.{bronze_table}"

print(f"PipelineRunId : {PipelineRunId}")
print(f"Bronze table  : {bronze_full_table}")
print(f"Entity        : {entity}")

## Create Silver target and quarantine schemas

In [ ]:
for schema_name in ["clinical", "reference", "population", "quarantine"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_LAKEHOUSE}.{schema_name}")


## Loading helpers

Entity transformations contain essential blocking field checks and quarantine invalid rows so only valid data reaches Silver. The standalone DQ notebook measures and reports those checks.


In [ ]:
def add_silver_metadata(
    df,
    pipeline_run_id,
    source_system,
    source_entity
):
    """
    Remove technical metadata inherited from the source/Bronze layer
    and create fresh Silver metadata.

    Silver keeps ONLY:
        _ingestion_timestamp
        _source_system
        _source_entity
        _pipeline_run_id
    """

    # ---------------------------------------------------------
    # Remove metadata coming from the source/Bronze table
    # ---------------------------------------------------------
    source_metadata_columns = [
        # Standardized Bronze metadata
        "_ingestion_timestamp",
        "_source_system",
        "_source_file",
        "_source_entity",
        "_pipeline_run_id",

        # Legacy metadata names
        "ingestion_timestamp",
        "source_system",
        "source_file",
        "source_entity",
        "pipeline_run_id",

        # Remove previous Silver-specific metadata, if present
        "_silver_processed_timestamp",
        "_silver_pipeline_run_id"
    ]

    columns_to_drop = [
        c for c in source_metadata_columns
        if c in df.columns
    ]

    if columns_to_drop:
        df = df.drop(*columns_to_drop)

    # ---------------------------------------------------------
    # Create NEW metadata for this Silver record
    # ---------------------------------------------------------
    return (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_system",
            F.lit(source_system)
        )
        .withColumn(
            "_source_entity",
            F.lit(source_entity)
        )
        .withColumn(
            "_pipeline_run_id",
            F.lit(pipeline_run_id)
        )
    )

def keep_latest(df, key_columns):
    """
    Keep the latest Bronze version of each business key.
    """
    w = (
        Window
        .partitionBy(*key_columns)
        .orderBy(F.col("_ingestion_timestamp").desc())
    )

    return (
        df
        .withColumn("_row_number", F.row_number().over(w))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

def split_valid_invalid(df, invalid_condition):
    """
    Split records into valid and invalid sets.
    """
    is_invalid = F.coalesce(invalid_condition, F.lit(False))
    invalid_df = df.filter(is_invalid)
    valid_df = df.filter(~is_invalid)
    return valid_df, invalid_df

def merge_to_silver(df, table_name, key_column):
    """
    Insert new rows and update existing rows in Silver.
    Validate source/target schema before MERGE.
    """

    if not spark.catalog.tableExists(table_name):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        print(f"Created Silver table: {table_name}")
        return

    target = DeltaTable.forName(
        spark,
        table_name
    )

    target_columns = set(
        target.toDF().columns
    )

    source_columns = set(
        df.columns
    )

    missing_in_source = (
        target_columns - source_columns
    )

    if missing_in_source:
        raise ValueError(
            f"Source is missing columns required by "
            f"{table_name}: "
            f"{sorted(missing_in_source)}"
        )

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            f"target.{key_column} = "
            f"source.{key_column}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged into Silver table: "
        f"{table_name}"
    )

def append_quarantine(base_df, new_df):
    """
    Add newly rejected rows to the entity quarantine dataframe.
    """
    return base_df.unionByName(
        new_df,
        allowMissingColumns=True
    )

def write_quarantine(df, table_name):
    """
    Write rejected rows to the entity quarantine table.

    The caller checks that rows exist before this function is called.
    """
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Quarantine records written to: {table_name}")

def apply_patient_reference_dq(
    entity,
    valid_df,
    quarantine_df
):
    """
    Check that PatientID exists in Silver Patient.

    Applied to:
      - Encounter  -> DQ004
      - Observation -> DQ007
    """

    rule_map = {
        "encounter": (
            "DQ004",
            "Encounter PatientID does not exist in Silver Patient"
        ),
        "observation": (
            "DQ007",
            "Observation PatientID does not exist in Silver Patient"
        )
    }

    if entity not in rule_map:
        return valid_df, quarantine_df

    patient_table = (
        f"{SILVER_LAKEHOUSE}.clinical.Patient"
    )

    if not spark.catalog.tableExists(patient_table):
        raise RuntimeError(
            f"{rule_map[entity][0]} cannot be checked because "
            f"{patient_table} does not exist. "
            "Process Patient before this entity."
        )

    patient_keys = (
        spark.table(patient_table)
        .select("PatientID")
        .distinct()
    )

    # Rows whose PatientID cannot be found.
    rejected_df = (
        valid_df
        .join(
            patient_keys,
            "PatientID",
            "leftanti"
        )
        .withColumn(
            "RuleID",
            F.lit(rule_map[entity][0])
        )
        .withColumn(
            "FailureReason",
            F.lit(rule_map[entity][1])
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    # Rows that have a valid PatientID.
    valid_df = (
        valid_df
        .join(
            patient_keys,
            "PatientID",
            "leftsemi"
        )
    )

    quarantine_df = append_quarantine(
        quarantine_df,
        rejected_df
    )

    return valid_df, quarantine_df

In [ ]:
def add_quarantine_metadata(df, pipeline_run_id):
    """Retain rejection information and stamp the loading run, without producing a DQ audit."""
    return df.withColumn("RunID", F.lit(pipeline_run_id))


## 7. Entity Transformation Functions


### 1. Organizations → Facility

In [ ]:
def transform_facility(df, pipeline_run_id):

    # ---------------------------------------------------------
    # 1. Rename / standardize columns
    # ---------------------------------------------------------
    facility_df = (
        df
        .withColumnRenamed("Id", "FacilityID")
        .withColumnRenamed("NAME", "FacilityName")
        .withColumnRenamed("ADDRESS", "Address")
        .withColumnRenamed("CITY", "City")
        .withColumnRenamed("STATE", "State")
        .withColumnRenamed("ZIP", "ZipCode")
        .withColumnRenamed("LAT", "Latitude")
        .withColumnRenamed("LON", "Longitude")
        .withColumnRenamed("PHONE", "Phone")
        .withColumnRenamed("REVENUE", "Revenue")
        .withColumnRenamed("UTILIZATION", "Utilization")
    )

    # ---------------------------------------------------------
    # 2. Enforce data types
    # ---------------------------------------------------------
    facility_df = (
        facility_df
        .withColumn("ZipCode", F.col("ZipCode").cast("string"))
        .withColumn("Latitude", F.col("Latitude").cast("double"))
        .withColumn("Longitude", F.col("Longitude").cast("double"))
        .withColumn("Revenue", F.col("Revenue").cast("double"))
        .withColumn("Utilization", F.col("Utilization").cast("double"))
    )

    # ---------------------------------------------------------
    # 3. Standardize values
    # ---------------------------------------------------------
    facility_df = (
        facility_df
        .withColumn("FacilityName", F.trim(F.col("FacilityName")))
        .withColumn("Address", F.trim(F.col("Address")))
        .withColumn("City", F.initcap(F.trim(F.col("City"))))
        .withColumn("State", F.upper(F.trim(F.col("State"))))
        .withColumn("Phone", F.trim(F.col("Phone")))
    )

    # ---------------------------------------------------------
    # 4. Silver metadata
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # 5. Deduplicate
    # ---------------------------------------------------------
    facility_df = keep_latest(
        facility_df,
        ["FacilityID"]
    )

    # ---------------------------------------------------------
    # 6. DQ rules
    # ---------------------------------------------------------
    invalid_condition = (
        F.col("FacilityID").isNull()
        |
        F.col("FacilityName").isNull()
        |
        (
            F.col("Latitude").isNotNull()
            &
            ~F.col("Latitude").between(-90, 90)
        )
        |
        (
            F.col("Longitude").isNotNull()
            &
            ~F.col("Longitude").between(-180, 180)
        )
        |
        (
            F.col("Revenue").isNotNull()
            &
            (F.col("Revenue") < 0)
        )
        |
        (
            F.col("Utilization").isNotNull()
            &
            (F.col("Utilization") < 0)
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        facility_df,
        invalid_condition
    )

    # ---------------------------------------------------------
    # 7. Quarantine information
    # ---------------------------------------------------------
    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_FACILITY"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("FacilityID").isNull(),
                "FacilityID is null"
            )
            .when(
                F.col("FacilityName").isNull(),
                "FacilityName is null"
            )
            .when(
                ~F.col("Latitude").between(-90, 90),
                "Invalid latitude"
            )
            .when(
                ~F.col("Longitude").between(-180, 180),
                "Invalid longitude"
            )
            .when(
                F.col("Revenue") < 0,
                "Revenue cannot be negative"
            )
            .when(
                F.col("Utilization") < 0,
                "Utilization cannot be negative"
            )
            .otherwise("Facility validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 2. Providers → Provider

In [ ]:
def transform_provider(df, pipeline_run_id):

    provider_df = (
        df
        .withColumnRenamed("Id", "ProviderID")
        .withColumnRenamed("ORGANIZATION", "FacilityID")
        .withColumnRenamed("NAME", "ProviderName")
        .withColumnRenamed("GENDER", "Gender")
        .withColumnRenamed("SPECIALITY", "Specialty")
        .withColumnRenamed("ADDRESS", "Address")
        .withColumnRenamed("CITY", "City")
        .withColumnRenamed("STATE", "State")
        .withColumnRenamed("ZIP", "ZipCode")
        .withColumnRenamed("LAT", "Latitude")
        .withColumnRenamed("LON", "Longitude")
        .withColumnRenamed("ENCOUNTERS", "EncounterCount")
        .withColumnRenamed("PROCEDURES", "ProcedureCount")
    )

    provider_df = (
        provider_df
        .withColumn("ZipCode", F.col("ZipCode").cast("string"))
        .withColumn("Latitude", F.col("Latitude").cast("double"))
        .withColumn("Longitude", F.col("Longitude").cast("double"))
        .withColumn("EncounterCount", F.col("EncounterCount").cast("long"))
        .withColumn("ProcedureCount", F.col("ProcedureCount").cast("long"))
    )

    provider_df = (
        provider_df
        .withColumn("ProviderName", F.trim(F.col("ProviderName")))
        .withColumn("Specialty", F.trim(F.col("Specialty")))
        .withColumn("City", F.initcap(F.trim(F.col("City"))))
        .withColumn("State", F.upper(F.trim(F.col("State"))))
        .withColumn(
            "Gender",
            F.when(
                F.upper(F.trim(F.col("Gender"))).isin("M", "MALE"),
                "Male"
            )
            .when(
                F.upper(F.trim(F.col("Gender"))).isin("F", "FEMALE"),
                "Female"
            )
            .otherwise("Unknown")
        )
    )


    provider_df = keep_latest(
        provider_df,
        ["ProviderID"]
    )

    invalid_condition = (
        F.col("ProviderID").isNull()
        |
        F.col("ProviderName").isNull()
        |
        (
            F.col("Latitude").isNotNull()
            &
            ~F.col("Latitude").between(-90, 90)
        )
        |
        (
            F.col("Longitude").isNotNull()
            &
            ~F.col("Longitude").between(-180, 180)
        )
        |
        (
            F.col("EncounterCount").isNotNull()
            &
            (F.col("EncounterCount") < 0)
        )
        |
        (
            F.col("ProcedureCount").isNotNull()
            &
            (F.col("ProcedureCount") < 0)
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        provider_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_PROVIDER"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("ProviderID").isNull(),
                "ProviderID is null"
            )
            .when(
                F.col("ProviderName").isNull(),
                "ProviderName is null"
            )
            .when(
                ~F.col("Latitude").between(-90, 90),
                "Invalid latitude"
            )
            .when(
                ~F.col("Longitude").between(-180, 180),
                "Invalid longitude"
            )
            .when(
                F.col("EncounterCount") < 0,
                "EncounterCount cannot be negative"
            )
            .when(
                F.col("ProcedureCount") < 0,
                "ProcedureCount cannot be negative"
            )
            .otherwise("Provider validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 3. Payers → Payer

In [ ]:
def transform_payer(df, pipeline_run_id):

    payer_df = (
        df
        .withColumnRenamed("Id", "PayerID")
        .withColumnRenamed("NAME", "PayerName")
        .withColumnRenamed("OWNERSHIP", "Ownership")
        .withColumnRenamed("ADDRESS", "Address")
        .withColumnRenamed("CITY", "City")
        .withColumnRenamed("STATE_HEADQUARTERED", "StateHeadquartered")
        .withColumnRenamed("ZIP", "ZipCode")
        .withColumnRenamed("PHONE", "Phone")
        .withColumnRenamed("AMOUNT_COVERED", "AmountCovered")
        .withColumnRenamed("AMOUNT_UNCOVERED", "AmountUncovered")
        .withColumnRenamed("REVENUE", "Revenue")
        .withColumnRenamed("COVERED_ENCOUNTERS", "CoveredEncounters")
        .withColumnRenamed("UNCOVERED_ENCOUNTERS", "UncoveredEncounters")
        .withColumnRenamed("COVERED_MEDICATIONS", "CoveredMedications")
        .withColumnRenamed("UNCOVERED_MEDICATIONS", "UncoveredMedications")
        .withColumnRenamed("COVERED_PROCEDURES", "CoveredProcedures")
        .withColumnRenamed("UNCOVERED_PROCEDURES", "UncoveredProcedures")
        .withColumnRenamed("COVERED_IMMUNIZATIONS", "CoveredImmunizations")
        .withColumnRenamed("UNCOVERED_IMMUNIZATIONS", "UncoveredImmunizations")
        .withColumnRenamed("UNIQUE_CUSTOMERS", "UniqueCustomers")
        .withColumnRenamed("QOLS_AVG", "AverageQOLS")
        .withColumnRenamed("MEMBER_MONTHS", "MemberMonths")
    )

    payer_df = (
        payer_df
        .withColumn("ZipCode", F.col("ZipCode").cast("string"))
        .withColumn("AmountCovered", F.col("AmountCovered").cast("double"))
        .withColumn("AmountUncovered", F.col("AmountUncovered").cast("double"))
        .withColumn("Revenue", F.col("Revenue").cast("double"))
        .withColumn("AverageQOLS", F.col("AverageQOLS").cast("double"))
    )

    count_columns = [
        "CoveredEncounters",
        "UncoveredEncounters",
        "CoveredMedications",
        "UncoveredMedications",
        "CoveredProcedures",
        "UncoveredProcedures",
        "CoveredImmunizations",
        "UncoveredImmunizations",
        "UniqueCustomers",
        "MemberMonths"
    ]

    for c in count_columns:
        payer_df = payer_df.withColumn(
            c,
            F.col(c).cast("long")
        )

    payer_df = (
        payer_df
        .withColumn("PayerName", F.trim(F.col("PayerName")))
        .withColumn("Ownership", F.trim(F.col("Ownership")))
        .withColumn("City", F.initcap(F.trim(F.col("City"))))
        .withColumn(
            "StateHeadquartered",
            F.upper(F.trim(F.col("StateHeadquartered")))
        )
    )


    payer_df = keep_latest(
        payer_df,
        ["PayerID"]
    )

    invalid_condition = (
        F.col("PayerID").isNull()
        |
        F.col("PayerName").isNull()
        |
        (
            F.col("AmountCovered").isNotNull()
            &
            (F.col("AmountCovered") < 0)
        )
        |
        (
            F.col("AmountUncovered").isNotNull()
            &
            (F.col("AmountUncovered") < 0)
        )
        |
        (
            F.col("Revenue").isNotNull()
            &
            (F.col("Revenue") < 0)
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        payer_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_PAYER"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("PayerID").isNull(),
                "PayerID is null"
            )
            .when(
                F.col("PayerName").isNull(),
                "PayerName is null"
            )
            .when(
                F.col("AmountCovered") < 0,
                "AmountCovered cannot be negative"
            )
            .when(
                F.col("AmountUncovered") < 0,
                "AmountUncovered cannot be negative"
            )
            .when(
                F.col("Revenue") < 0,
                "Revenue cannot be negative"
            )
            .otherwise("Payer validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 4. Patients → Patient

In [ ]:
def transform_patient(df, pipeline_run_id):

    # ---------------------------------------------------------
    # 1. Rename / standardize columns
    # ---------------------------------------------------------
    patient_df = (
        df
        .withColumnRenamed("ID", "PatientID")
        .withColumnRenamed("BIRTHDATE", "BirthDate")
        .withColumnRenamed("DEATHDATE", "DeathDate")
        .withColumnRenamed("FIRST", "FirstName")
        .withColumnRenamed("LAST", "LastName")
        .withColumnRenamed("GENDER", "Gender")
        .withColumnRenamed("RACE", "Race")
        .withColumnRenamed("ETHNICITY", "Ethnicity")
        .withColumnRenamed("CITY", "City")
        .withColumnRenamed("ZIP", "ZipCode")
        .withColumnRenamed("ADDRESS", "Address")
        .withColumnRenamed("BIRTHPLACE", "BirthPlace")
        .withColumnRenamed("DRIVERS", "Drivers")
        .withColumnRenamed("PASSPORT", "Passport")
        .withColumnRenamed("MARITAL", "MaritalStatus")
        .withColumnRenamed("PREFIX", "Prefix")
        .withColumnRenamed("SUFFIX", "Suffix")
        .withColumnRenamed("MAIDEN", "Maiden")
    )

    # ---------------------------------------------------------
    # 2. Enforce data types
    # ---------------------------------------------------------
    patient_df = (
        patient_df
        .withColumn(
            "BirthDate",
            F.to_date(F.col("BirthDate"))
        )
        .withColumn(
            "DeathDate",
            F.to_date(F.col("DeathDate"))
        )
    )

    # ---------------------------------------------------------
    # 3. Standardize selected values
    # ---------------------------------------------------------
    patient_df = (
        patient_df
        .withColumn(
            "Gender",
            F.when(
                F.upper(F.trim(F.col("Gender"))).isin("M", "MALE"),
                "Male"
            )
            .when(
                F.upper(F.trim(F.col("Gender"))).isin("F", "FEMALE"),
                "Female"
            )
            .otherwise("Unknown")
        )
    )

    # ---------------------------------------------------------
    # 4. Add Silver processing metadata
    # Bronze ingestion metadata is preserved unchanged.
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # 5. Calculate age
    # ---------------------------------------------------------
    patient_df = patient_df.withColumn(
        "Age",
        F.when(
            F.col("BirthDate").isNotNull(),
            F.floor(
                F.months_between(
                    F.current_date(),
                    F.col("BirthDate")
                ) / 12
            )
        )
    )

    # ---------------------------------------------------------
    # 6. Calculate age group
    # ---------------------------------------------------------
    patient_df = patient_df.withColumn(
        "AgeGroup",
        F.when(F.col("Age").isNull(), "Unknown")
        .when(F.col("Age") <= 4, "0-4")
        .when(F.col("Age") <= 17, "5-17")
        .when(F.col("Age") <= 34, "18-34")
        .when(F.col("Age") <= 49, "35-49")
        .when(F.col("Age") <= 64, "50-64")
        .otherwise("65+")
    )

    # ---------------------------------------------------------
    # 7. Deduplicate
    # Keep latest ORIGINAL Bronze ingestion.
    # ---------------------------------------------------------
    patient_df = keep_latest(
        patient_df,
        ["PatientID"]
    )

    # ---------------------------------------------------------
    # 8. Define DQ rules
    # ---------------------------------------------------------
    invalid_condition = (
        F.col("PatientID").isNull()
        |
        (
            F.col("BirthDate").isNotNull()
            &
            (F.col("BirthDate") > F.current_date())
        )
        |
        (
            F.col("DeathDate").isNotNull()
            &
            F.col("BirthDate").isNotNull()
            &
            (F.col("DeathDate") < F.col("BirthDate"))
        )
    )

    # ---------------------------------------------------------
    # 9. Separate valid / invalid records
    # ---------------------------------------------------------
    valid_df, invalid_df = split_valid_invalid(
        patient_df,
        invalid_condition
    )

    # ---------------------------------------------------------
    # 10. Add DQ rule information to rejected records
    #
    # Each rejected Patient gets the first blocking rule that failed.
    # DQ009 (Gender) is only a WARNING, so it is logged later and
    # does not reject the Patient.
    # ---------------------------------------------------------
    quarantine_df = (
        invalid_df
        .withColumn(
            "RuleID",
            F.when(
                F.col("PatientID").isNull(),
                F.lit("DQ001")
            )
            .when(
                F.col("BirthDate") > F.current_date(),
                F.lit("DQ002")
            )
            .when(
                F.col("DeathDate") < F.col("BirthDate"),
                F.lit("DQ003")
            )
        )
        .withColumn(
            "Severity",
            F.when(
                F.col("RuleID") == "DQ001",
                F.lit("CRITICAL")
            )
            .otherwise(
                F.lit("ERROR")
            )
        )
        .withColumn(
            "FailureReason",
            F.when(
                F.col("RuleID") == "DQ001",
                F.lit("PatientID is null")
            )
            .when(
                F.col("RuleID") == "DQ002",
                F.lit("BirthDate is in the future")
            )
            .when(
                F.col("RuleID") == "DQ003",
                F.lit("DeathDate is before BirthDate")
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df


### 5. Encounters → Encounter

In [ ]:
def transform_encounter(df, pipeline_run_id):

    # ---------------------------------------------------------
    # 1. Rename columns
    # ---------------------------------------------------------
    encounter_df = (
        df
        .withColumnRenamed("ID", "EncounterID")
        .withColumnRenamed("DATE", "EncounterDate")
        .withColumnRenamed("PATIENT", "PatientID")
        .withColumnRenamed("CODE", "EncounterCode")
        .withColumnRenamed("DESCRIPTION", "EncounterDescription")
        .withColumnRenamed("REASONCODE", "ReasonCode")
        .withColumnRenamed("REASONDESCRIPTION", "ReasonDescription")
    )

    # ---------------------------------------------------------
    # 2. Enforce data types
    # ---------------------------------------------------------
    encounter_df = (
        encounter_df
        .withColumn(
            "EncounterDate",
            F.to_timestamp(F.col("EncounterDate"))
        )
        .withColumn(
            "EncounterCode",
            F.col("EncounterCode").cast("string")
        )
        .withColumn(
            "ReasonCode",
            F.col("ReasonCode").cast("string")
        )
    )

    # ---------------------------------------------------------
    # 3. Standardize text
    # ---------------------------------------------------------
    encounter_df = (
        encounter_df
        .withColumn(
            "EncounterDescription",
            F.trim(F.col("EncounterDescription"))
        )
        .withColumn(
            "ReasonDescription",
            F.trim(F.col("ReasonDescription"))
        )
    )

    # ---------------------------------------------------------
    # 4. Add Silver metadata
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # 5. DQ010 - detect duplicate EncounterID
    #
    # The latest duplicate is kept.
    # Older duplicates are sent to quarantine.
    # Null EncounterID rows are not removed here; they continue
    # to the normal validation below.
    # ---------------------------------------------------------
    non_null_ids = encounter_df.filter(
        F.col("EncounterID").isNotNull()
    )

    null_ids = encounter_df.filter(
        F.col("EncounterID").isNull()
    )

    rank_window = (
        Window
        .partitionBy("EncounterID")
        .orderBy(F.col("_ingestion_timestamp").desc())
    )

    ranked = (
        non_null_ids
        .withColumn(
            "_EncounterRank",
            F.row_number().over(rank_window)
        )
    )

    duplicate_df = (
        ranked
        .filter(F.col("_EncounterRank") > 1)
        .drop("_EncounterRank")
        .withColumn("RuleID", F.lit("DQ010"))
        .withColumn("Severity", F.lit("CRITICAL"))
        .withColumn(
            "FailureReason",
            F.lit(
                "Duplicate EncounterID; latest record retained"
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    latest_df = (
        ranked
        .filter(F.col("_EncounterRank") == 1)
        .drop("_EncounterRank")
    )

    # Add null-ID rows back so they can be rejected by validation.
    encounter_df = latest_df.unionByName(
        null_ids,
        allowMissingColumns=True
    )

    # ---------------------------------------------------------
    # 6. General Encounter validation
    # ---------------------------------------------------------
    invalid_condition = (
        F.col("EncounterID").isNull()
        |
        F.col("PatientID").isNull()
        |
        F.col("EncounterDate").isNull()
        |
        (
            F.col("EncounterDate")
            > F.current_timestamp()
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        encounter_df,
        invalid_condition
    )

    invalid_df = (
        invalid_df
        .withColumn(
            "RuleID",
            F.lit("DQ_ENCOUNTER")
        )
        .withColumn(
            "Severity",
            F.lit("ERROR")
        )
        .withColumn(
            "FailureReason",
            F.when(
                F.col("EncounterID").isNull(),
                F.lit("EncounterID is null")
            )
            .when(
                F.col("PatientID").isNull(),
                F.lit("PatientID is null")
            )
            .when(
                F.col("EncounterDate").isNull(),
                F.lit("EncounterDate is null or invalid")
            )
            .when(
                F.col("EncounterDate") > F.current_timestamp(),
                F.lit("EncounterDate is in the future")
            )
            .otherwise(
                F.lit("Encounter validation failed")
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    # Combine normal validation failures and duplicate failures.
    quarantine_df = append_quarantine(
        invalid_df,
        duplicate_df
    )

    return valid_df, quarantine_df


### 6. Conditions → Condition

`ConditionID` is generated at the Silver layer; it does not exist in Bronze.

In [ ]:
def transform_condition(df, pipeline_run_id):

    condition_df = (
        df
        .withColumnRenamed("START", "StartDate")
        .withColumnRenamed("STOP", "StopDate")
        .withColumnRenamed("PATIENT", "PatientID")
        .withColumnRenamed("ENCOUNTER", "EncounterID")
        .withColumnRenamed("CODE", "ConditionCode")
        .withColumnRenamed("DESCRIPTION", "ConditionDescription")
    )

    condition_df = (
        condition_df
        .withColumn(
            "StartDate",
            F.to_date(F.col("StartDate"))
        )
        .withColumn(
            "StopDate",
            F.to_date(F.col("StopDate"))
        )
        .withColumn(
            "ConditionCode",
            F.col("ConditionCode").cast("string")
        )
    )

    # ---------------------------------------------------------
    # SILVER-GENERATED ID
    # ConditionID does not exist in Bronze.
    # Deterministic so pipeline reruns generate the same ID.
    # ---------------------------------------------------------
    condition_df = condition_df.withColumn(
        "ConditionID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PatientID"), F.lit("")),
                F.coalesce(F.col("EncounterID"), F.lit("")),
                F.coalesce(F.col("ConditionCode"), F.lit("")),
                F.coalesce(F.col("StartDate").cast("string"), F.lit("")),
                F.coalesce(F.col("StopDate").cast("string"), F.lit(""))
            ),
            256
        )
    )


    condition_df = keep_latest(
        condition_df,
        ["ConditionID"]
    )

    invalid_condition = (
        F.col("PatientID").isNull()
        |
        F.col("ConditionCode").isNull()
        |
        F.col("StartDate").isNull()
        |
        (
            F.col("StartDate") > F.current_date()
        )
        |
        (
            F.col("StopDate").isNotNull()
            &
            (F.col("StopDate") < F.col("StartDate"))
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        condition_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_CONDITION"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("PatientID").isNull(),
                "PatientID is null"
            )
            .when(
                F.col("ConditionCode").isNull(),
                "ConditionCode is null"
            )
            .when(
                F.col("StartDate").isNull(),
                "StartDate is null or invalid"
            )
            .when(
                F.col("StartDate") > F.current_date(),
                "StartDate is in the future"
            )
            .when(
                F.col("StopDate") < F.col("StartDate"),
                "StopDate is before StartDate"
            )
            .otherwise("Condition validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 7. Procedures → Procedure

`ProcedureID` is generated at the Silver layer; it does not exist in Bronze.

In [ ]:
def transform_procedure(df, pipeline_run_id):

    procedure_df = (
        df
        .withColumnRenamed("DATE", "ProcedureDate")
        .withColumnRenamed("PATIENT", "PatientID")
        .withColumnRenamed("ENCOUNTER", "EncounterID")
        .withColumnRenamed("CODE", "ProcedureCode")
        .withColumnRenamed("DESCRIPTION", "ProcedureDescription")
        .withColumnRenamed("REASONCODE", "ReasonCode")
        .withColumnRenamed("REASONDESCRIPTION", "ReasonDescription")
    )

    procedure_df = (
        procedure_df
        .withColumn(
            "ProcedureDate",
            F.to_timestamp(F.col("ProcedureDate"))
        )
        .withColumn(
            "ProcedureCode",
            F.col("ProcedureCode").cast("string")
        )
        .withColumn(
            "ReasonCode",
            F.col("ReasonCode").cast("string")
        )
    )

    # SILVER-GENERATED ID — not present in Bronze
    procedure_df = procedure_df.withColumn(
        "ProcedureID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PatientID"), F.lit("")),
                F.coalesce(F.col("EncounterID"), F.lit("")),
                F.coalesce(F.col("ProcedureCode"), F.lit("")),
                F.coalesce(F.col("ProcedureDate").cast("string"), F.lit(""))
            ),
            256
        )
    )


    procedure_df = keep_latest(
        procedure_df,
        ["ProcedureID"]
    )

    invalid_condition = (
        F.col("PatientID").isNull()
        |
        F.col("ProcedureDate").isNull()
        |
        F.col("ProcedureCode").isNull()
        |
        (
            F.col("ProcedureDate") >
            F.current_timestamp()
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        procedure_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_PROCEDURE"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("PatientID").isNull(),
                "PatientID is null"
            )
            .when(
                F.col("ProcedureDate").isNull(),
                "ProcedureDate is null or invalid"
            )
            .when(
                F.col("ProcedureCode").isNull(),
                "ProcedureCode is null"
            )
            .when(
                F.col("ProcedureDate") > F.current_timestamp(),
                "ProcedureDate is in the future"
            )
            .otherwise("Procedure validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 8. Medications → Medication

`MedicationID` is generated at the Silver layer; it does not exist in Bronze.

In [ ]:
def transform_medication(df, pipeline_run_id):

    medication_df = (
        df
        .withColumnRenamed("START", "StartDate")
        .withColumnRenamed("STOP", "StopDate")
        .withColumnRenamed("PATIENT", "PatientID")
        .withColumnRenamed("ENCOUNTER", "EncounterID")
        .withColumnRenamed("CODE", "MedicationCode")
        .withColumnRenamed("DESCRIPTION", "MedicationDescription")
        .withColumnRenamed("REASONCODE", "ReasonCode")
        .withColumnRenamed("REASONDESCRIPTION", "ReasonDescription")
    )

    medication_df = (
        medication_df
        .withColumn("StartDate", F.to_date(F.col("StartDate")))
        .withColumn("StopDate", F.to_date(F.col("StopDate")))
        .withColumn(
            "MedicationCode",
            F.col("MedicationCode").cast("string")
        )
        .withColumn(
            "ReasonCode",
            F.col("ReasonCode").cast("string")
        )
    )

    # SILVER-GENERATED ID — not present in Bronze
    medication_df = medication_df.withColumn(
        "MedicationID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PatientID"), F.lit("")),
                F.coalesce(F.col("EncounterID"), F.lit("")),
                F.coalesce(F.col("MedicationCode"), F.lit("")),
                F.coalesce(F.col("StartDate").cast("string"), F.lit("")),
                F.coalesce(F.col("StopDate").cast("string"), F.lit(""))
            ),
            256
        )
    )


    medication_df = keep_latest(
        medication_df,
        ["MedicationID"]
    )

    invalid_condition = (
        F.col("PatientID").isNull()
        |
        F.col("MedicationCode").isNull()
        |
        F.col("StartDate").isNull()
        |
        (
            F.col("StartDate") > F.current_date()
        )
        |
        (
            F.col("StopDate").isNotNull()
            &
            (F.col("StopDate") < F.col("StartDate"))
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        medication_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_MEDICATION"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("PatientID").isNull(),
                "PatientID is null"
            )
            .when(
                F.col("MedicationCode").isNull(),
                "MedicationCode is null"
            )
            .when(
                F.col("StartDate").isNull(),
                "StartDate is null or invalid"
            )
            .when(
                F.col("StartDate") > F.current_date(),
                "StartDate is in the future"
            )
            .when(
                F.col("StopDate") < F.col("StartDate"),
                "StopDate is before StartDate"
            )
            .otherwise("Medication validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 9. Observations → Observation

`ObservationID` is generated at the Silver layer; it does not exist in Bronze.

In [ ]:
def transform_observation(df, pipeline_run_id):

    observation_df = (
        df
        .withColumnRenamed("DATE", "ObservationDate")
        .withColumnRenamed("PATIENT", "PatientID")
        .withColumnRenamed("ENCOUNTER", "EncounterID")
        .withColumnRenamed("CODE", "ObservationCode")
        .withColumnRenamed("DESCRIPTION", "ObservationDescription")
        .withColumnRenamed("VALUE", "ObservationValue")
        .withColumnRenamed("UNITS", "Units")
    )

    observation_df = (
        observation_df
        .withColumn(
            "ObservationDate",
            F.to_timestamp(F.col("ObservationDate"))
        )
        .withColumn(
            "ObservationCode",
            F.col("ObservationCode").cast("string")
        )
        .withColumn(
            "ObservationValue",
            F.trim(F.col("ObservationValue"))
        )
        .withColumn(
            "Units",
            F.trim(F.col("Units"))
        )
    )

    # Optional numeric representation.
    # Text ObservationValue is still preserved.
    observation_df = observation_df.withColumn(
        "NumericValue",
        F.col("ObservationValue").cast("double")
    )

    # SILVER-GENERATED ID — not present in Bronze
    observation_df = observation_df.withColumn(
        "ObservationID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PatientID"), F.lit("")),
                F.coalesce(F.col("EncounterID"), F.lit("")),
                F.coalesce(F.col("ObservationCode"), F.lit("")),
                F.coalesce(
                    F.col("ObservationDate").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("ObservationValue"), F.lit("")),
                F.coalesce(F.col("Units"), F.lit(""))
            ),
            256
        )
    )


    observation_df = keep_latest(
        observation_df,
        ["ObservationID"]
    )

    invalid_condition = (
        F.col("PatientID").isNull()
        |
        F.col("ObservationDate").isNull()
        |
        F.col("ObservationCode").isNull()
        |
        (
            F.col("ObservationDate") >
            F.current_timestamp()
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        observation_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn("RuleID", F.lit("DQ_OBSERVATION"))
        .withColumn(
            "FailureReason",
            F.when(
                F.col("PatientID").isNull(),
                "PatientID is null"
            )
            .when(
                F.col("ObservationDate").isNull(),
                "ObservationDate is null or invalid"
            )
            .when(
                F.col("ObservationCode").isNull(),
                "ObservationCode is null"
            )
            .when(
                F.col("ObservationDate") > F.current_timestamp(),
                "ObservationDate is in the future"
            )
            .otherwise("Observation validation failed")
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 10. Rural/Urban → RuralUrban

`RuralUrbanID` is generated at the Silver layer; FIPS is retained as a code.

In [ ]:
def transform_rural_urban(df, pipeline_run_id):

    rural_df = (
        df
        .withColumnRenamed("State", "State")
        .withColumnRenamed("County_Name", "CountyName")
        .withColumnRenamed("Attribute", "Attribute")
        .withColumnRenamed("Value", "AttributeValue")
    )

    rural_df = (
        rural_df
        .withColumn(
            "FIPS",
            F.lpad(
                F.col("FIPS").cast("string"),
                5,
                "0"
            )
        )
        .withColumn(
            "State",
            F.trim(F.col("State"))
        )
        .withColumn(
            "CountyName",
            F.trim(F.col("CountyName"))
        )
        .withColumn(
            "Attribute",
            F.trim(F.col("Attribute"))
        )
        .withColumn(
            "AttributeValue",
            F.trim(F.col("AttributeValue"))
        )
    )

    # SILVER-GENERATED ID
    # FIPS alone is not sufficient because each FIPS can have
    # multiple Attribute rows.
    rural_df = rural_df.withColumn(
        "RuralUrbanID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("FIPS"), F.lit("")),
                F.coalesce(F.col("Attribute"), F.lit(""))
            ),
            256
        )
    )


    rural_df = keep_latest(
        rural_df,
        ["RuralUrbanID"]
    )

    invalid_condition = (
        F.col("FIPS").isNull()
        |
        F.col("CountyName").isNull()
        |
        F.col("Attribute").isNull()
    )

    valid_df, invalid_df = split_valid_invalid(
        rural_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn(
            "RuleID",
            F.lit("DQ_RURAL_URBAN")
        )
        .withColumn(
            "FailureReason",
            F.when(
                F.col("FIPS").isNull(),
                "FIPS is null"
            )
            .when(
                F.col("CountyName").isNull(),
                "CountyName is null"
            )
            .when(
                F.col("Attribute").isNull(),
                "Attribute is null"
            )
            .otherwise(
                "Rural/Urban validation failed"
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 11. USDA Income → USDAIncome

In [ ]:
def transform_usda_income(df, pipeline_run_id):

    income_df = (
        df
        .withColumnRenamed(
            "Median_HH_Inc_ACS",
            "MedianHouseholdIncomeACS"
        )
        .withColumnRenamed(
            "PerCapitaInc",
            "PerCapitaIncome"
        )
        .withColumnRenamed(
            "Poverty_Rate_0_17_ACS",
            "ChildPovertyRateACS"
        )
        .withColumnRenamed(
            "Poverty_Rate_ACS",
            "PovertyRateACS"
        )
        .withColumnRenamed(
            "Deep_Pov_All",
            "DeepPovertyAll"
        )
        .withColumnRenamed(
            "Deep_Pov_Children",
            "DeepPovertyChildren"
        )
        .withColumnRenamed(
            "NumAll_inPOV_ACS",
            "PopulationInPovertyACS"
        )
        .withColumnRenamed(
            "PCTPOV017",
            "ChildPovertyPercent"
        )
        .withColumnRenamed(
            "POV017",
            "ChildrenInPoverty"
        )
        .withColumnRenamed(
            "MedHHInc",
            "MedianHouseholdIncome"
        )
        .withColumnRenamed(
            "POVALL",
            "PopulationInPoverty"
        )
        .withColumnRenamed(
            "PCTPOVALL",
            "PovertyPercent"
        )
        .withColumnRenamed(
            "Num_inPOV_0_17_ACS",
            "ChildrenInPovertyACS"
        )
    )

    income_df = (
        income_df
        .withColumn(
            "FIPS",
            F.lpad(
                F.col("FIPS").cast("string"),
                5,
                "0"
            )
        )
        .withColumn("State", F.trim(F.col("State")))
        .withColumn("County", F.trim(F.col("County")))
    )

    numeric_columns = [
        "MedianHouseholdIncomeACS",
        "PerCapitaIncome",
        "ChildPovertyRateACS",
        "PovertyRateACS",
        "DeepPovertyAll",
        "DeepPovertyChildren",
        "PopulationInPovertyACS",
        "ChildPovertyPercent",
        "ChildrenInPoverty",
        "MedianHouseholdIncome",
        "PopulationInPoverty",
        "PovertyPercent",
        "ChildrenInPovertyACS"
    ]

    for c in numeric_columns:
        income_df = income_df.withColumn(
            c,
            F.col(c).cast("double")
        )


    income_df = keep_latest(
        income_df,
        ["FIPS"]
    )

    invalid_condition = (
        F.col("FIPS").isNull()
        |
        F.col("State").isNull()
        |
        F.col("County").isNull()
        |
        (
            F.col("MedianHouseholdIncomeACS").isNotNull()
            &
            (F.col("MedianHouseholdIncomeACS") < 0)
        )
        |
        (
            F.col("PerCapitaIncome").isNotNull()
            &
            (F.col("PerCapitaIncome") < 0)
        )
    )

    valid_df, invalid_df = split_valid_invalid(
        income_df,
        invalid_condition
    )

    quarantine_df = (
        invalid_df
        .withColumn(
            "RuleID",
            F.lit("DQ_USDA_INCOME")
        )
        .withColumn(
            "FailureReason",
            F.when(
                F.col("FIPS").isNull(),
                "FIPS is null"
            )
            .when(
                F.col("State").isNull(),
                "State is null"
            )
            .when(
                F.col("County").isNull(),
                "County is null"
            )
            .when(
                F.col("MedianHouseholdIncomeACS") < 0,
                "Median household income cannot be negative"
            )
            .when(
                F.col("PerCapitaIncome") < 0,
                "Per-capita income cannot be negative"
            )
            .otherwise(
                "USDA income validation failed"
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )

    return valid_df, quarantine_df

### 12. bronze_usda_jobs → usda_job


In [ ]:


def transform_usda_jobs(df, pipeline_run_id):

    jobs_df = df

    # ============================================================
    # 1. Standardize column names
    # ============================================================

    rename_map = {}

    for col_name in jobs_df.columns:

        # UnempRate2021 -> UnemploymentRate2021
        if re.fullmatch(r"UnempRate\d{4}", col_name):
            year = col_name[-4:]
            rename_map[col_name] = f"UnemploymentRate{year}"

        # PctEmpChange2021 -> EmploymentChangePct2021
        # PctEmpChange1920 -> EmploymentChangePct1920
        elif re.fullmatch(r"PctEmpChange\d+", col_name):
            period = col_name.replace("PctEmpChange", "")
            rename_map[col_name] = f"EmploymentChangePct{period}"

        # Employment-sector percentages
        elif col_name.startswith("PctEmp"):

            suffix = col_name.replace("PctEmp", "")

            sector_map = {
                "Agriculture": "AgricultureEmploymentPct",
                "Mining": "MiningEmploymentPct",
                "Construction": "ConstructionEmploymentPct",
                "Manufacturing": "ManufacturingEmploymentPct",
                "Trade": "TradeEmploymentPct",
                "Trans": "TransportationEmploymentPct",
                "Information": "InformationEmploymentPct",
                "FIRE": "FinanceInsuranceRealEstateEmploymentPct",
                "Services": "ServicesEmploymentPct",
                "Govt": "GovernmentEmploymentPct"
            }

            if suffix in sector_map:
                rename_map[col_name] = sector_map[suffix]

        # NumCivLaborForce2021 / NumCivLaborforce2021
        elif re.fullmatch(
            r"NumCivLabor[Ff]orce\d{4}",
            col_name
        ):
            year = col_name[-4:]
            rename_map[col_name] = f"CivilianLaborForce{year}"

        # NumUnemployed2021 -> Unemployed2021
        elif re.fullmatch(r"NumUnemployed\d{4}", col_name):
            year = col_name[-4:]
            rename_map[col_name] = f"Unemployed{year}"

        # NumEmployed2021 -> Employed2021
        elif re.fullmatch(r"NumEmployed\d{4}", col_name):
            year = col_name[-4:]
            rename_map[col_name] = f"Employed{year}"

        # NumCivEmployed -> CivilianEmployed
        elif col_name == "NumCivEmployed":
            rename_map[col_name] = "CivilianEmployed"


    for old_name, new_name in rename_map.items():
        jobs_df = jobs_df.withColumnRenamed(
            old_name,
            new_name
        )


    # ============================================================
    # 2. Standardize geography
    # IMPORTANT: FIPS is an identifier, not a numeric measure
    # ============================================================

    jobs_df = (
        jobs_df
        .withColumn(
            "FIPS",
            F.lpad(
                F.col("FIPS").cast("string"),
                5,
                "0"
            )
        )
        .withColumn(
            "State",
            F.trim(F.col("State"))
        )
        .withColumn(
            "County",
            F.trim(F.col("County"))
        )
    )


    # ============================================================
    # 3. Enforce numeric data types
    # ============================================================

    rate_columns = [
        c for c in jobs_df.columns
        if c.startswith("UnemploymentRate")
    ]

    change_columns = [
        c for c in jobs_df.columns
        if c.startswith("EmploymentChangePct")
    ]

    employment_pct_columns = [
        c for c in jobs_df.columns
        if c.endswith("EmploymentPct")
    ]

    count_columns = [
        c for c in jobs_df.columns
        if (
            c.startswith("Employed")
            or c.startswith("Unemployed")
            or c.startswith("CivilianLaborForce")
            or c == "CivilianEmployed"
        )
    ]


    # Percent/rate measures
    for c in (
        rate_columns
        + change_columns
        + employment_pct_columns
    ):
        jobs_df = jobs_df.withColumn(
            c,
            F.col(c).cast("double")
        )


    # Population/employment counts
    #
    # Source currently stores many of these as FLOAT.
    # Cast to LONG in Silver because they represent counts.
    for c in count_columns:
        jobs_df = jobs_df.withColumn(
            c,
            F.col(c).cast("long")
        )


    # ============================================================
    # 4. Add Silver metadata
    # Keep original Bronze metadata unchanged
    # ============================================================



    # ============================================================
    # 5. Deduplicate
    # One Silver record per county/FIPS
    # ============================================================

    jobs_df = keep_latest(
        jobs_df,
        ["FIPS"]
    )


    # ============================================================
    # 6. Data-quality rules
    # ============================================================

    invalid_condition = (
        F.col("FIPS").isNull()
        |
        F.col("State").isNull()
        |
        F.col("County").isNull()
    )


    # Unemployment rates should normally be 0-100
    for c in rate_columns:

        invalid_condition = (
            invalid_condition
            |
            (
                F.col(c).isNotNull()
                &
                (
                    (F.col(c) < 0)
                    |
                    (F.col(c) > 100)
                )
            )
        )


    # Employment sector percentages should be 0-100
    for c in employment_pct_columns:

        invalid_condition = (
            invalid_condition
            |
            (
                F.col(c).isNotNull()
                &
                (
                    (F.col(c) < 0)
                    |
                    (F.col(c) > 100)
                )
            )
        )


    # Counts cannot be negative
    for c in count_columns:

        invalid_condition = (
            invalid_condition
            |
            (
                F.col(c).isNotNull()
                &
                (F.col(c) < 0)
            )
        )


    valid_df, invalid_df = split_valid_invalid(
        jobs_df,
        invalid_condition
    )


    # ============================================================
    # 7. Quarantine information
    # ============================================================

    failure_reason = (
        F.when(
            F.col("FIPS").isNull(),
            "FIPS is null"
        )
        .when(
            F.col("State").isNull(),
            "State is null"
        )
        .when(
            F.col("County").isNull(),
            "County is null"
        )
    )


    # Identify invalid unemployment rates
    for c in rate_columns:

        failure_reason = failure_reason.when(
            F.col(c).isNotNull()
            &
            (
                (F.col(c) < 0)
                |
                (F.col(c) > 100)
            ),
            F.lit(
                f"{c} must be between 0 and 100"
            )
        )


    # Identify invalid sector percentages
    for c in employment_pct_columns:

        failure_reason = failure_reason.when(
            F.col(c).isNotNull()
            &
            (
                (F.col(c) < 0)
                |
                (F.col(c) > 100)
            ),
            F.lit(
                f"{c} must be between 0 and 100"
            )
        )


    # Identify negative counts
    for c in count_columns:

        failure_reason = failure_reason.when(
            F.col(c).isNotNull()
            &
            (F.col(c) < 0),
            F.lit(
                f"{c} cannot be negative"
            )
        )


    quarantine_df = (
        invalid_df
        .withColumn(
            "RuleID",
            F.lit("DQ_USDA_JOBS")
        )
        .withColumn(
            "FailureReason",
            failure_reason.otherwise(
                "USDA jobs validation failed"
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )


    return valid_df, quarantine_df

### 13.  bronze_usda_people → usda_people

In [ ]:
def transform_usda_people(df, pipeline_run_id):

    people_df = df

    # ============================================================
    # 1. Standardize geography
    # ============================================================

    people_df = (
        people_df
        .withColumn(
            "FIPS",
            F.lpad(
                F.col("FIPS").cast("string"),
                5,
                "0"
            )
        )
        .withColumn(
            "State",
            F.trim(F.col("State"))
        )
        .withColumn(
            "County",
            F.trim(F.col("County"))
        )
    )


    # ============================================================
    # 2. Identify percentage / rate columns
    # These remain DOUBLE.
    #
    # NOTE:
    # Population-change and migration rates may legitimately
    # be negative, so we do NOT apply a 0-100 rule to all rates.
    # ============================================================

    percentage_columns = [
        "Under18Pct2020",
        "Age65AndOlderPct2020",
        "WhiteNonHispanicPct2020",
        "BlackNonHispanicPct2020",
        "AsianNonHispanicPct2020",
        "NativeAmericanNonHispanicPct2020",
        "HispanicPct2020",
        "MultipleRacePct2020",
        "ForeignBornPct",
        "ForeignBornEuropePct",
        "ForeignBornMexPct",
        "NonEnglishHHPct",
        "Ed1LessThanHSPct",
        "Ed2HSDiplomaOnlyPct",
        "Ed3SomeCollegePct",
        "Ed4AssocDegreePct",
        "Ed5CollegePlusPct",
        "FemaleHHPct",
        "HH65PlusAlonePct",
        "OwnHomePct",
        "ForeignBornCentralSouthAmPct",
        "ForeignBornCaribPct",
        "ForeignBornAsiaPct",
        "ForeignBornAfricaPct"
    ]


    rate_columns = [
        "Pop_change_Rate_2020_2021",
        "Net_Migration_Rate_2020_2021",
        "Natural_Change_Rate_2020_2021",
        "Net_InterMigrationRate_2020_2021",
        "PopChangeRate1020",
        "NonHispWhitePopChangeRate1020",
        "NonHispBlackPopChangeRate1020",
        "NonHispAsianPopChangeRate1020",
        "NonHispNatAmerPopChangeRate1020",
        "HispanicPopChangeRate1020",
        "MultipleRacePopChangeRate1020",
        "Net_International_Migration_Rate_2010_2019",
        "NaturalChangeRate1019",
        "NetMigrationRate1019",
        "PopChangeRate1019"
    ]


    other_double_columns = [
        "PopDensity2020",
        "AvgHHSize",
        "LandAreaSQMiles2020"
    ]


    for c in (
        percentage_columns
        + rate_columns
        + other_double_columns
    ):
        if c in people_df.columns:
            people_df = people_df.withColumn(
                c,
                F.col(c).cast("double")
            )


    # ============================================================
    # 3. Population / count measures
    #
    # Bronze stores many of these as DOUBLE, but they represent
    # people/household counts. Convert them to LONG in Silver.
    # ============================================================

    count_columns = [
        "POPESTIMATE2021",

        "WhiteNonHispanicNum2020",
        "BlackNonHispanicNum2020",
        "AsianNonHispanicNum2020",
        "NativeAmericanNonHispanicNum2020",
        "HispanicNum2020",
        "MultipleRaceNum2020",

        "Ed3SomeCollegeNum",
        "Ed2HSDiplomaOnlyNum",
        "Ed1LessThanHSNum",

        "TotalPopACS",
        "ForeignBornNum",
        "TotalPop25Plus",
        "ForeignBornEuropeNum",
        "ForeignBornCaribNum",
        "ForeignBornMexNum",
        "Ed5CollegePlusNum",
        "HH65PlusAloneNum",
        "Ed4AssocDegreeNum",
        "ForeignBornAfricaNum",
        "ForeignBornAsiaNum",
        "NonEnglishHHNum",
        "OwnHomeNum",
        "FemaleHHNum",
        "TotalHH",
        "ForeignBornCentralSouthAmNum",
        "TotalOccHU",

        "Age65AndOlderNum2020",
        "Under18Num2020",
        "TotalPop2020",

        "NaturalChange1019",

        "TotalPopEst2015",
        "TotalPopEst2016",
        "TotalPopEst2017",
        "TotalPopEst2018",
        "TotalPopEst2019",
        "TotalPopEst2012",
        "TotalPopEst2014",
        "TotalPopEst2013",
        "TotalPopEstBase2010",
        "TotalPopEst2011",
        "TotalPopEst2010",

        "Net_International_Migration_2010_2019",
        "NetMigration1019",

        "ESTIMATESBASE2020",
        "POPESTIMATE2020",
        "NaturalChg2020_2021",
        "Net_InterMigration_2020_2021",
        "Net_Migration_2020_2021"
    ]


    for c in count_columns:
        if c in people_df.columns:
            people_df = people_df.withColumn(
                c,
                F.col(c).cast("long")
            )


    # ============================================================
    # 4. Add Silver metadata
    #
    # Bronze metadata is retained:
    #   _ingestion_timestamp
    #   _source_system
    #   _source_entity
    #   _pipeline_run_id
    #
    # These fields are specifically created in Silver.
    # ============================================================



    # ============================================================
    # 5. Deduplicate
    # One county-level record per FIPS.
    # Latest Bronze ingestion wins.
    # ============================================================

    people_df = keep_latest(
        people_df,
        ["FIPS"]
    )


    # ============================================================
    # 6. Core DQ rules
    # ============================================================

    invalid_condition = (
        F.col("FIPS").isNull()
        |
        F.col("State").isNull()
        |
        F.col("County").isNull()
    )


    # ------------------------------------------------------------
    # Population / household counts cannot be negative
    # ------------------------------------------------------------

    for c in count_columns:

        if c in people_df.columns:

            invalid_condition = (
                invalid_condition
                |
                (
                    F.col(c).isNotNull()
                    &
                    (F.col(c) < 0)
                )
            )


    # ------------------------------------------------------------
    # Demographic percentages must be between 0 and 100.
    #
    # Do NOT apply this to population-change/migration rates,
    # because those can legitimately be negative.
    # ------------------------------------------------------------

    for c in percentage_columns:

        if c in people_df.columns:

            invalid_condition = (
                invalid_condition
                |
                (
                    F.col(c).isNotNull()
                    &
                    (
                        (F.col(c) < 0)
                        |
                        (F.col(c) > 100)
                    )
                )
            )


    # ------------------------------------------------------------
    # Physical measures
    # ------------------------------------------------------------

    if "PopDensity2020" in people_df.columns:
        invalid_condition = (
            invalid_condition
            |
            (
                F.col("PopDensity2020").isNotNull()
                &
                (F.col("PopDensity2020") < 0)
            )
        )

    if "LandAreaSQMiles2020" in people_df.columns:
        invalid_condition = (
            invalid_condition
            |
            (
                F.col("LandAreaSQMiles2020").isNotNull()
                &
                (F.col("LandAreaSQMiles2020") < 0)
            )
        )

    if "AvgHHSize" in people_df.columns:
        invalid_condition = (
            invalid_condition
            |
            (
                F.col("AvgHHSize").isNotNull()
                &
                (F.col("AvgHHSize") < 0)
            )
        )


    # ============================================================
    # 7. Separate valid / invalid records
    # ============================================================

    valid_df, invalid_df = split_valid_invalid(
        people_df,
        invalid_condition
    )


    # ============================================================
    # 8. Build meaningful quarantine reason
    # ============================================================

    failure_reason = (
        F.when(
            F.col("FIPS").isNull(),
            "FIPS is null"
        )
        .when(
            F.col("State").isNull(),
            "State is null"
        )
        .when(
            F.col("County").isNull(),
            "County is null"
        )
    )


    # Percentage validation reasons
    for c in percentage_columns:

        if c in people_df.columns:

            failure_reason = failure_reason.when(
                F.col(c).isNotNull()
                &
                (
                    (F.col(c) < 0)
                    |
                    (F.col(c) > 100)
                ),
                F.lit(
                    f"{c} must be between 0 and 100"
                )
            )


    # Negative count reasons
    for c in count_columns:

        if c in people_df.columns:

            failure_reason = failure_reason.when(
                F.col(c).isNotNull()
                &
                (F.col(c) < 0),
                F.lit(
                    f"{c} cannot be negative"
                )
            )


    # Physical measure checks
    if "PopDensity2020" in people_df.columns:

        failure_reason = failure_reason.when(
            F.col("PopDensity2020") < 0,
            "PopDensity2020 cannot be negative"
        )


    if "LandAreaSQMiles2020" in people_df.columns:

        failure_reason = failure_reason.when(
            F.col("LandAreaSQMiles2020") < 0,
            "LandAreaSQMiles2020 cannot be negative"
        )


    if "AvgHHSize" in people_df.columns:

        failure_reason = failure_reason.when(
            F.col("AvgHHSize") < 0,
            "AvgHHSize cannot be negative"
        )


    quarantine_df = (
        invalid_df
        .withColumn(
            "RuleID",
            F.lit("DQ_USDA_PEOPLE")
        )
        .withColumn(
            "FailureReason",
            failure_reason.otherwise(
                "USDA people validation failed"
            )
        )
        .withColumn(
            "FailureTimestamp",
            F.current_timestamp()
        )
    )


    return valid_df, quarantine_df

## 8. Transformer Registry


In [ ]:
TRANSFORMERS = {
    "patient": transform_patient,
    "organization": transform_facility,
    "provider": transform_provider,
    "payer": transform_payer,
    "encounter": transform_encounter,
    "condition": transform_condition,
    "procedure": transform_procedure,
    "medication": transform_medication,
    "observation": transform_observation,
    "rural_urban_2023": transform_rural_urban,
    "usda_income": transform_usda_income,
    "usda_job": transform_usda_jobs,
    "usda_people": transform_usda_people
    
}

## 9. Silver Target Configuration


In [ ]:
ENTITY_CONFIG = {
    "patient": {
        "silver_table": "clinical.Patient",
        "key": "PatientID"
    },
    "organization": {
        "silver_table": "reference.Facility",
        "key": "FacilityID"
    },
    "provider": {
        "silver_table": "reference.Provider",
        "key": "ProviderID"
    },
    "payer": {
        "silver_table": "reference.Payer",
        "key": "PayerID"
    },
    "encounter": {
        "silver_table": "clinical.Encounter",
        "key": "EncounterID"
    },
    "condition": {
        "silver_table": "clinical.Condition",
        "key": "ConditionID"
    },
    "procedure": {
        "silver_table": "clinical.Procedure",
        "key": "ProcedureID"
    },
    "medication": {
        "silver_table": "clinical.Medication",
        "key": "MedicationID"
    },
    "observation": {
        "silver_table": "clinical.Observation",
        "key": "ObservationID"
    },
    "rural_urban_2023": {
        "silver_table": "population.RuralUrban",
        "key": "RuralUrbanID"
    },
    "usda_income": {
        "silver_table": "population.USDAIncome",
        "key": "FIPS"
    },
    "usda_job": {
        "silver_table": "population.usda_job",
        "key": "FIPS",
        "processing_mode": "full"
    },
    "usda_people": {
        "silver_table": "population.USDAPeople",
        "key": "FIPS",
        "processing_mode": "full"
    }
}

## 10. Validate Entity and Read Bronze


In [ ]:
if entity not in TRANSFORMERS:
    raise ValueError(
        f"No Silver transformation is registered for EntityName='{EntityName}' "
        f"(logical entity='{entity}')."
    )

if entity not in ENTITY_CONFIG:
    raise ValueError(
        f"No Silver target configuration is registered for entity='{entity}'."
    )

config = ENTITY_CONFIG[entity]

bronze_df = spark.read.table(bronze_full_table)

print(f"Loaded Bronze source: {bronze_full_table}")
print(f"Bronze columns ({len(bronze_df.columns)}): {bronze_df.columns}")

## Execute Bronze → Silver loading

In [ ]:
# Transform and apply blocking rules already defined inside the transformers.
valid_df, quarantine_df = TRANSFORMERS[entity](bronze_df, PipelineRunId)
valid_df, quarantine_df = apply_patient_reference_dq(entity, valid_df, quarantine_df)

silver_entity_name = config["silver_table"].split(".")[-1]
silver_table = f"{SILVER_LAKEHOUSE}.{config['silver_table']}"
quarantine_table = f"{SILVER_LAKEHOUSE}.quarantine.{silver_entity_name}"

# The original Bronze timestamp is available to deduplication before stamping Silver metadata.
valid_df = add_silver_metadata(valid_df, PipelineRunId, SourceName, silver_entity_name)
quarantine_df = add_silver_metadata(quarantine_df, PipelineRunId, SourceName, silver_entity_name)
quarantine_df = add_quarantine_metadata(quarantine_df, PipelineRunId)

valid_df = valid_df.persist(StorageLevel.MEMORY_AND_DISK)
quarantine_df = quarantine_df.persist(StorageLevel.MEMORY_AND_DISK)
try:
    valid_count = valid_df.count()
    rejected_count = quarantine_df.count()
    if valid_count:
        merge_to_silver(valid_df, silver_table, config["key"])
    if rejected_count:
        write_quarantine(quarantine_df, quarantine_table)
    print(f"Loading complete: {silver_table}; accepted={valid_count:,}; quarantined={rejected_count:,}; run={PipelineRunId}")
finally:
    valid_df.unpersist()
    quarantine_df.unpersist()
